# Qwen PRM Scoring Smoke Test

Smoke test for `Qwen2.5-Math-PRM-7B`. The PRM scores reasoning steps
at `<extra_0>` separator tokens via a dedicated reward head that emits
an (incorrect, correct) probability pair at each separator.

Two examples: (1) the model-card flamingo problem, to check scores
match the published reference; (2) a correct-vs-wrong algebra pair, to
check the PRM flags a known mistake.

Note: the model card specifies `bfloat16`, but the V100 (sm_70) has no
bf16 support, so we load `float16`. fp16 preserves step *rankings* but
can drift the absolute scores, so don't expect an exact match. See the
`bfloat16-vs-float16-on-v100` findings note.

Env: runs under `py311` (transformers 4.57). The bundled remote code
(`modeling_qwen2_rm.py`) calls a cache API removed in newer
transformers, so the forward pass uses `use_cache=False` — correct for
single-pass scoring and it sidesteps the incompatibility.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

from notebook_utils import gpu_mem_used_gb, print_step_scores

base_dir = "/groups/chichengz/tnn/datasets"
qwen_prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

## System prompt

Shared by both examples; matches the model card.

In [2]:
# System prompt per the Qwen2.5-Math-PRM-7B model card.
# Shared by both examples. Double backslash so "\boxed" is a
# literal backslash, not a "\b" backspace.
qwen_system_prompt = (
    "Please reason step by step, and put your final answer "
    "within \\boxed{}."
)

## Scoring function

Builds the chat prompt with a trailing `<extra_0>` after every step,
runs one forward pass, and reads `P(correct)` at each separator. Set
`print_conversation=True` to dump the exact PRM input.

In [ ]:
def score_qwen_prm(
    model,
    tokenizer,
    problem: str,
    steps: list[str],
    system: str,
    step_separator: str = "<extra_0>",
    print_conversation: bool = False,
) -> list[float]:
    # step_separator is the reserved token Qwen2.5-Math-PRM-7B
    # uses to mark step boundaries; the PRM head emits an
    # (incorrect, correct) probability pair at each occurrence.
    # The trailing separator gives the last step its own score
    # position.
    assistant = step_separator.join(steps) + step_separator
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": assistant},
    ]
    # add_generation_prompt=False: keep the assistant turn
    # ending at the steps as written instead of appending a
    # "your turn" cue.
    conversation = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    if print_conversation:
        print("===== Formatted PRM input =====")
        print(conversation)
        print("===============================")
    input_ids = tokenizer.encode(
        conversation, return_tensors="pt"
    ).to(model.device)

    # add_special_tokens=False: avoid prepending BOS so we get
    # just the single token id for the separator literal.
    sep_ids = tokenizer.encode(
        step_separator, add_special_tokens=False
    )
    if len(sep_ids) != 1:
        raise ValueError(f"Expected one separator token, got {sep_ids}")
    sep_positions = (input_ids[0] == sep_ids[0]).nonzero(as_tuple=True)[0]
    # Guard against tokenizer merges that change the separator
    # count.
    if sep_positions.numel() != len(steps):
        raise RuntimeError(
            f"Expected {len(steps)} separators, "
            f"found {sep_positions.numel()}"
        )

    # use_cache=False: one full-sequence forward pass needs no
    # KV cache. It also sidesteps the bundled remote code's
    # get_usable_length() call, removed in transformers >=4.5x
    # (the crash under py311 / transformers 4.57).
    with torch.no_grad():
        logits = model(input_ids=input_ids, use_cache=False)[0]

    # PRM head emits 2 logits per token; index 1 is P(correct).
    probs = F.softmax(logits, dim=-1)
    return probs[0, sep_positions, 1].detach().cpu().float().tolist()

## Load the PRM

In [4]:
def load_qwen_prm(
    model_dir: str,
    device_map: str = "cuda:0",
):
    # float16 for V100 (sm_70); model card says bfloat16
    # (Ampere+), so scores may differ slightly on Ampere GPUs.
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, trust_remote_code=True
    )
    model = AutoModel.from_pretrained(
        model_dir,
        device_map=device_map,
        dtype=torch.float16,
        trust_remote_code=True,
    ).eval()
    return model, tokenizer


qwen_model, qwen_tokenizer = load_qwen_prm(qwen_prm_dir)

print(f"model dtype: {next(qwen_model.parameters()).dtype}")
print(f"GPU memory : {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

model dtype: torch.float16
GPU memory : 13.87 GB


## Example 1 — model-card flamingo problem

Expected scores (bf16, model card): `[1.0, 0.1904, 0.9766, 1.0]`. On
V100/fp16 we expect close, not exact.

In [5]:
# Toy example from the Qwen2.5-Math-PRM-7B model card.
# Double-backslash the LaTeX (\\times, \\boxed) so it survives
# intact — a single "\t"/"\b" would become a tab/backspace
# control char and corrupt the input.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [6]:
qwen_scores = score_qwen_prm(
    qwen_model, qwen_tokenizer, problem, reasoning_steps,
    qwen_system_prompt,
)

print("=== Flamingo trajectory ===")
print_step_scores(reasoning_steps, qwen_scores)

=== Flamingo trajectory ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1580
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...


## Example 2 — correct vs. wrong trajectory

Same equation scored twice. The wrong trajectory divides by 2 instead
of 3 at step 2 — a good PRM should drop the score there and stay low.
The first run prints the exact PRM input for inspection.

In [7]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [8]:
# Inspect the exact PRM input for the correct trajectory:
# every step should end with <extra_0>, including the final
# answer step. Then score both trajectories.
correct_scores = score_qwen_prm(
    qwen_model, qwen_tokenizer, algebra_problem, correct_steps,
    qwen_system_prompt, print_conversation=True,
)
wrong_scores = score_qwen_prm(
    qwen_model, qwen_tokenizer, algebra_problem, wrong_steps,
    qwen_system_prompt,
)

print("\n=== Correct trajectory ===")
print_step_scores(correct_steps, correct_scores)

print("\n=== Wrong trajectory (step 2 divides by 2) ===")
print_step_scores(wrong_steps, wrong_scores)

===== Formatted PRM input =====
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
If 3x + 5 = 17, what is x?<|im_end|>
<|im_start|>assistant
We need solve the equation 3x + 5 = 17.<extra_0>Subtracting 5 from both sides gives 3x = 12.<extra_0>Dividing both sides by 3 gives x = 4.<extra_0>Therefore, the answer is (\boxed{4}).<extra_0><|im_end|><|endoftext|>

=== Correct trajectory ===
Step 1: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).

=== Wrong trajectory (step 2 divides by 2) ===
Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0103
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.5132
Therefore, the final answer is \boxed{6}.


## Cleanup

In [9]:
del qwen_model, qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory: {gpu_mem_used_gb():.2f} GB")

GPU memory: 13.57 GB
